# Week 20, Day 6: Forge: Week 20 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. log2 6 =", round(np.log2(6), 3))
print("2. 0.5 log2(16) =", 0.5 * np.log2(16))
print("3.", round(0.5 * np.log2(0.5 / 0.25) + 0.5 * np.log2(0.5 / 0.75), 4))

In [ ]:
rng = np.random.default_rng(1999)
def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]
    return -np.sum(p * np.log2(p))
def plugin_mi(s, r, S, R):
    J = np.zeros((S, R)); np.add.at(J, (s, r), 1); J /= J.sum()
    return entropy(J.sum(1)) + entropy(J.sum(0)) - entropy(J.ravel())
def corrected_mi(s, r, S, R, rng, n_shuf=200):
    return plugin_mi(s, r, S, R) - np.mean([plugin_mi(rng.permutation(s), r, S, R) for _ in range(n_shuf)])

S, n_per = 4, 50
stim = np.repeat(np.arange(S), n_per)
latency = 10 + 10 * stim + 4 * rng.normal(size=len(stim))          # ms
count = rng.poisson(6, len(stim))                                  # the same mean count for every stimulus

### Exercise: Part A: the information is in the timing

Bin the latency into 5 ms bins from 0 to 60 ms (12 bins, clipping), cap the count at 15, and compute the shuffle-corrected information of each, storing `info_count` and `info_latency`.

In [ ]:
lat_bin = ...
cnt_bin = ...
info_count = corrected_mi(stim, cnt_bin, S, 16, rng)
info_latency = ...
print(f"information in the count {info_count:.3f} bits, in the first-spike latency {info_latency:.3f} bits (maximum 2)")

In [ ]:
import numpy as np
def test_count():
    "the corrected count information is close to zero (below 0.1 bits)"
    assert info_count < 0.1

def test_latency():
    "the corrected latency information is above 1 bit (out of a possible 2)"
    assert info_latency > 1.0

test_count()
test_latency()
print("All checks passed")

### Exercise: Part B: two neurons, each silent alone

Build the joint table of S (rows, 2 values) against the pair (R1, R2) (4 columns) for equally likely S and R1, with R2 = S XOR R1. Compute `i_pair`, and the information of each neuron alone (`i_r1`, `i_r2`) by summing the table over the other neuron.

In [ ]:
J = np.zeros((2, 4))                     # columns index r1 * 2 + r2
for s_ in (0, 1):
    for r1 in (0, 1):
        r2 = s_ ^ r1
        J[s_, r1 * 2 + r2] += 0.25
def mi_table(T): return entropy(T.sum(1)) + entropy(T.sum(0)) - entropy(T.ravel())
i_pair = mi_table(J)
J_r1 = ...                               # 2 x 2: S against R1
J_r2 = ...                               # 2 x 2: S against R2
i_r1, i_r2 = mi_table(J_r1), mi_table(J_r2)
print(f"I(S; R1) = {i_r1:.3f}, I(S; R2) = {i_r2:.3f}, I(S; R1, R2) = {i_pair:.3f} bits")

In [ ]:
import numpy as np
def test_alone():
    "each neuron alone carries no information about S"
    assert abs(i_r1) < 1e-12 and abs(i_r2) < 1e-12

def test_together():
    "together they carry the full 1 bit"
    assert abs(i_pair - 1) < 1e-12

test_alone()
test_together()
print("All checks passed")

### Exercise: Part C: the CDF curve transmits more

Draw 200,000 inputs from a gamma distribution (shape 2), compute both responses (linear: x scaled so the 99.5th percentile is 1 and clipped; CDF: the empirical CDF), add noise, and estimate the information with a 40 × 40 joint histogram of input quantile and response (plug-in; the bias is small with this many samples). Store the values in `info['linear']` and `info['cdf']`.

In [ ]:
r_c = np.random.default_rng(1981)
x = r_c.gamma(2.0, 1.0, 200000)
g = {"linear": np.clip(x / np.percentile(x, 99.5), 0, 1),
     "cdf": np.searchsorted(np.sort(x), x) / len(x)}
x_bin = np.minimum((np.searchsorted(np.sort(x), x) / len(x) * 40).astype(int), 39)    # input in 40 equal-probability bins
info = {}
for name, resp in g.items():
    r = resp + 0.08 * r_c.normal(size=len(x))
    r_bin = ...                                  # response in 40 equal-width bins between -0.3 and 1.3, clipped
    info[name] = plugin_mi(x_bin, r_bin, 40, 40)
print({k: round(v, 3) for k, v in info.items()}, " bits")

In [ ]:
import numpy as np
def test_cdf_better():
    "the CDF-matched response carries at least 0.3 bits more than the linear one"
    assert info["cdf"] > info["linear"] + 0.3

def test_bound():
    "both stay below the capacity of a 0 to 1 output with noise sd 0.08, about log2(1 / (0.08 sqrt(2 pi e))) + 0.5"
    assert all(v < np.log2(1 / (0.08 * np.sqrt(2 * np.pi * np.e))) + 0.5 for v in info.values())

test_cdf_better()
test_bound()
print("All checks passed")